# Track #2 — RT-DETR-l  ·  Kaggle (T4 ×2)

**Setup (right sidebar) before running:**
1. **Accelerator** → `GPU T4 x2`  ·  **Internet** → `On`
2. **Add Input → Competitions** → add this competition (data auto-found under `/kaggle/input/`). *No upload needed.*
3. In **cell 1**, set `REPO_URL` to your GitHub repo. *(Repo name / folder layout don't matter — scripts are auto-located.)*

**How to run:** Run cells **1→4** (setup + smoke test). Paste me the smoke-test output to confirm.
Then run the **one** 🚀 heavy cell (cell 5) and go to sleep — it trains all 3 folds **and** writes
`submission_rtdetr.csv`, refreshing it **after every fold** (so a valid CSV always exists, even if the
session stops early). Cell 6 is only needed if you want to (re)generate the CSV separately.


## 1 · Clone repo + install  *(edit REPO_URL)*


In [ ]:
REPO_URL = 'https://github.com/<YOUR_USER>/<YOUR_REPO>.git'   # <<< EDIT THIS ONLY

import os, sys, glob, shutil, subprocess
REPO_DIR = '/kaggle/working/' + REPO_URL.rstrip('/').split('/')[-1].replace('.git','')
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)          # always pull the latest code
subprocess.run(['git','clone','--depth','1',REPO_URL,REPO_DIR], check=True)
# auto-locate scripts dir (works whatever the repo is named / structured)
hits = glob.glob(f'{REPO_DIR}/**/rtdetr_lib.py', recursive=True)
assert hits, f'rtdetr_lib.py not found under {REPO_DIR} -- is REPO_URL correct?'
SCRIPTS = os.path.dirname(hits[0])
if SCRIPTS not in sys.path: sys.path.insert(0, SCRIPTS)
for m in ('rtdetr_lib','submission_utils','ensemble_tracks'):  # drop any stale cached import
    sys.modules.pop(m, None)
print('scripts:', SCRIPTS)

subprocess.run(['pip','-q','install','ultralytics','albumentations'], check=True)
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)
print('CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


## 2 · Config + build dataset


In [ ]:
import rtdetr_lib as L

SRC      = L.find_competition_src()      # auto-finds /kaggle/input/<comp> (has train/train.csv)
DS_OUT   = '/kaggle/working/yolo_ds'
RUNS     = '/kaggle/working/runs'
OUT      = '/kaggle/working'
TEST_DIR = f'{SRC}/test/images'

MODEL    = 'rtdetr-l.pt'   # COCO-pretrained, auto-downloads (Internet must be ON)
K        = 3
IMGSZ    = 1280           # max small-object quality; fits T4x2. (Fallback if OOM: 1024)
BATCH    = 4              # T4x2 total @1280 (2/GPU). RT-DETR transformer is memory-heavy --
                          #   batch 8 (like Track#1's YOLO) risks OOM at 1280. Read smoke-test
                          #   'peak=<GB>': <=7GB/GPU -> BATCH=8 ok (keep headroom for 38-box frames);
                          #   OOM -> IMGSZ=1024 (then BATCH=8 fits) or BATCH=2.
DEVICE   = '0,1'          # BOTH GPUs (DDP). Single GPU -> '0'
EPOCHS   = 100
PATIENCE = 20
FOLDS    = [0, 1, 2]      # all three train automatically, one after another
print('SRC =', SRC)

DS = L.build_dataset(SRC, DS_OUT, k=K)


## 3 · 🔬 SMOKE TEST (~2-3 min) — run this, then send me the output
Trains 2 epochs on 64 images at the **real** IMGSZ/BATCH/DEVICE, then does a test prediction.
It prints peak GPU memory so we know 1280 fits. **If it OOMs:** set `IMGSZ=1024` (or `BATCH=2`)
in cell 2 and re-run cells 2→3.


In [ ]:
smoke_yaml = L.make_smoke_subset(DS, n_train=64, n_val=24)
L.train_one(smoke_yaml, RUNS, 'smoke', model=MODEL, imgsz=IMGSZ, batch=BATCH, device=DEVICE, smoke=True)
c = L.predict_one(f'{RUNS}/smoke/weights/best.pt', TEST_DIR, imgsz=IMGSZ, device='0')
print('inference path OK — predicted', len(c), 'test images')


## 4 · 🚀 HEAVY CELL — run once, then sleep
Trains **all** folds in `FOLDS` back-to-back. After **each** fold finishes it (re)writes
`/kaggle/working/submission_rtdetr.csv` using every fold completed so far (WBF-fused).
So if the session dies mid-way you still have the best submission available at that point.
Per-fold failures are caught and skipped — they won't abort the others.


In [ ]:
done = []
for k in FOLDS:
    print(f'\n=============== TRAIN FOLD {k} ===============')
    try:
        L.train_one(f'{DS}/data_fold{k}.yaml', RUNS, f'rtdetr_l_f{k}', model=MODEL,
                    imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS, patience=PATIENCE, device=DEVICE)
        done.append(k)
    except Exception as e:
        print(f'[WARN] fold {k} failed: {e!r} — continuing')
        continue
    # refresh submission from all folds done so far (cached folds are reused, cheap)
    try:
        sub = L.infer_all_folds(RUNS, TEST_DIR, OUT, imgsz=IMGSZ, device='0')
        L.validate_submission(sub, TEST_DIR)
        print(f'>>> submission refreshed using folds {done} -> {sub}')
    except Exception as e:
        print(f'[WARN] inference after fold {k} failed: {e!r} — will retry next fold')
print(f'\nDONE. folds trained: {done}. Submission: {OUT}/submission_rtdetr.csv')


## 5 · (only if needed) regenerate the CSV from saved weights
Run this on wake-up if the heavy cell got interrupted *after* training but before writing the CSV.
Uses whatever `best.pt` files exist — no retraining.


In [ ]:
sub = L.infer_all_folds(RUNS, TEST_DIR, OUT, imgsz=IMGSZ, device='0')
L.validate_submission(sub, TEST_DIR)
import pandas as pd; print(); print(pd.read_csv(sub).head(3).to_string())


## 6 · Submit
Download **`/kaggle/working/submission_rtdetr.csv`** and submit it. ✅ Complete Track-2 entry.

Per-fold prediction caches are in `/kaggle/working/caches/` for the optional grand ensemble with
Track #1 later (`track2/scripts/ensemble_tracks.py` — accepts Track #1's `submission.csv` directly).
